# Leverage-Voronoi landmarks -- full A/B experiments -> paper artifacts

Runs every experiment TWICE: with the default **segment-mean** landmarks and with the new
**leverage-seeded Voronoi-mean** landmarks (`landmark_mode=1`), plus the **full-attention (sdpa)**
ceiling where it is affordable. The only thing that changes between the two Nystrom arms is the
landmark selector -- identical pinv, `kappa_star=0`, identical recipe -- so any gap is the selector.

Branch: `leverage-landmarks`. Run top to bottom. Every cell is **resumable** (skips a run whose JSON
already exists) and streams to a `.log`, so a disconnect loses nothing.

Rough cost on one A100 (3 seeds): MQAR ~45 min, CIFAR ~15 min, STL@2304 ~75 min, STL@9216 ~2.5 h.
Trim `SEEDS`, drop the STL@9216 cell, or lower `epochs` to go faster.

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name} sm_{cap[0]}{cap[1]}. Switch runtime.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code (leverage-landmarks branch) + CUTLASS submodule

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
REPO_DIR = '/content/flashnystrom'   # absolute repo root; every cell cd's here
BRANCH   = 'leverage-landmarks'
%cd /content
!rm -rf flashnystrom
!git clone --recurse-submodules -b $BRANCH $REPO_URL flashnystrom
%cd {REPO_DIR}
!cd "{REPO_DIR}" && git submodule update --init --recursive
import os
assert os.path.isdir(f'{REPO_DIR}/third_party/cutlass/include'), \
    'CUTLASS submodule did not fetch -- re-run this cell (check network).'
print('on branch', BRANCH, '- CUTLASS headers present - ok to build')

## 2. Build the fused CUDA kernels (~3-6 min)

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
# Colab's toolchain trips -Werror on third-party headers; LAX disables that guard.
os.environ['FLASH_NYSTROM_LAX_BUILD'] = '1'
!cd "{REPO_DIR}" && pip install -e . --no-build-isolation

## 2a. Assert the build imported (fail loudly)
If the extension did not compile, or you built from the wrong branch, this stops the run here
with a clear message instead of a confusing failure 20 minutes into training.

In [ ]:
import flash_nystrom
import flash_nystrom._C as _C
assert hasattr(_C, 'forward') and hasattr(_C, 'backward'), 'core kernels missing from _C'
assert hasattr(_C, 'debug_leverage_landmarks'), \
    'leverage-landmark binding missing -- _C built from the wrong branch or a stale build. '\
    'Re-run the clone + build cells (must be branch leverage-landmarks).'
# confirm the mode-1 wiring is present end to end at the C++ boundary
import torch
_q = torch.randn(1, 1, 128, 64, device='cuda', dtype=torch.float16)
_out = _C.forward(_q, _q, _q, 64, 6, 0.0, False, 1, 0, 1)  # ..., landmark_mode=1, seed=0, subsample=1
assert len(_out) == 17 and torch.isfinite(_out[0]).all(), 'mode-1 forward did not return finite output'
print('build OK: _C imported, leverage bindings present, mode-1 forward callable ('
      f'{torch.cuda.get_device_name()})')

## 3. Verify the leverage path (kernel + straight-through backward)

In [ ]:
# numerical + autograd tests for the new landmark selector, then a live mode-1 smoke
!cd "{REPO_DIR}" && python -m pytest tests/test_leverage_landmarks.py tests/test_leverage_forward_backward.py -q
import torch
from flash_nystrom import flash_nystrom_attention
mk = lambda: torch.randn(4, 2, 512, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
qg = q.clone().requires_grad_(True)
o = flash_nystrom_attention(qg, k, v, 64, 6, landmark_mode=1)
o.sum().backward()
print('mode-1 fwd finite:', bool(torch.isfinite(o).all()),
      '| bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Config + output dir
Drive is optional: if it mounts, results persist there; otherwise they go to a local folder.
Datasets are downloaded once and cached (to Drive when available).

In [ ]:
import os, shutil, torchvision
RUN_NAME = 'leverage_ab'          # change for a separate run
SEEDS = [0, 1, 2]
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/flashnystrom_runs/' + RUN_NAME
    DATA_CACHE = '/content/drive/MyDrive/flashnystrom_runs/data_cache'
except Exception as e:
    print('no Drive (', e, ') -- using local ./', RUN_NAME)
    OUTDIR = os.path.join(REPO_DIR, RUN_NAME); DATA_CACHE = None
RAW = OUTDIR + '/raw'
os.makedirs(RAW, exist_ok=True)
print('All results ->', OUTDIR)

# datasets: download once, restore from cache each session
DATA_DIR = '/content/data'; os.makedirs(DATA_DIR, exist_ok=True)
os.environ['FN_DATA_DIR'] = DATA_DIR
torchvision.datasets.CIFAR10.url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'
_TARBALLS = ['cifar-10-python.tar.gz', 'stl10_binary.tar.gz']
if DATA_CACHE:
    os.makedirs(DATA_CACHE, exist_ok=True)
    for f in _TARBALLS:
        s_, d_ = os.path.join(DATA_CACHE, f), os.path.join(DATA_DIR, f)
        if os.path.exists(s_) and not os.path.exists(d_): shutil.copy(s_, d_)
torchvision.datasets.CIFAR10(DATA_DIR, download=True)
torchvision.datasets.STL10(DATA_DIR, split='train', download=True)
if DATA_CACHE:
    for f in _TARBALLS:
        s_, d_ = os.path.join(DATA_DIR, f), os.path.join(DATA_CACHE, f)
        if os.path.exists(s_) and not os.path.exists(d_): shutil.copy(s_, d_)
print('datasets ready in', DATA_DIR)

## 5. MQAR recall A/B  ->  `raw/mqar_{backend}_seed{S}.json`
Associative recall: the task where landmark quality matters most. Backends: full attention (ceiling),
segment-mean Nystrom, leverage-Voronoi Nystrom. `kappa_star=0` (short context -> K2 well-conditioned).

In [ ]:
import re, json
MQAR_BACKENDS = ['sdpa', 'flash_nystrom', 'flash_nystrom_leverage']
for backend in MQAR_BACKENDS:
    for s in SEEDS:
        out = f'{RAW}/mqar_{backend}_seed{s}.json'
        if os.path.exists(out): print('skip', out); continue
        log = f'{RAW}/mqar_{backend}_seed{s}.log'
        cmd = ('python -m paper.mqar.train'
               f' --backend {backend} --seed {s} --kappa_star 0'
               ' --seq_len 256 --num_kv_pairs 16 --num_landmarks 64'
               ' --newton_iter 6 --grad_clip 1.0 --batch_size 256'
               f' 2>&1 | tee {log}')
        !cd "{REPO_DIR}" && {cmd}
        txt = open(log).read()
        mm = re.search(r'best test recall:\s*([\d.]+)%', txt)
        json.dump({'experiment': 'mqar', 'backend': backend, 'seed': s,
                   'recall': float(mm.group(1)) if mm else None},
                  open(out, 'w'), indent=2)
print('mqar done')

## 6. CIFAR-10 ViT A/B (N=65, cheap sanity)  ->  `raw/cifar_lev_seed{S}.json`
m=64 ~ N=65, so landmarks ~ tokens and the two selectors should tie -- a control that confirms the
leverage gain is specific to the m<<N regime, not a free lunch everywhere.

In [ ]:
for s in SEEDS:
    out = f'{RAW}/cifar_lev_seed{s}.json'
    if os.path.exists(out): print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset cifar10 --patch_size 4 --epochs 20 --autobatch --grad_clip 1.0 --kappa_star 0'
           f' --seed {s} --backends sdpa flash_nystrom_vanilla flash_nystrom_leverage flash_nystrom_leverage_det'
           f' --out_json {out} 2>&1 | tee {RAW}/cifar_lev_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}
print('cifar done')

## 7. STL-10 @ 2304 tokens A/B (patch 2)  ->  `raw/stl10_p2_lev_seed{S}.json`
m=64 << N=2304: the regime where segment means blur many tokens per landmark and leverage selection
should help. sdpa ceiling included (affordable at this N).

In [ ]:
for s in SEEDS:
    out = f'{RAW}/stl10_p2_lev_seed{s}.json'
    if os.path.exists(out): print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset stl10 --patch_size 2 --epochs 50 --autobatch --grad_clip 1.0 --kappa_star 0'
           f' --seed {s} --backends sdpa flash_nystrom_vanilla flash_nystrom_leverage flash_nystrom_leverage_det'
           f' --out_json {out} 2>&1 | tee {RAW}/stl10_p2_lev_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}
print('stl@2304 done')

## 8. STL-10 @ 9216 tokens A/B (patch 1, extreme N)  ->  `raw/stl10_p1_lev_seed{S}.json`
sdpa excluded (O(N^2) memory). `--autobatch` (coordinated: see the batch note below). The hardest test of
landmark quality: 64 landmarks summarizing 9216 tokens.

In [ ]:
for s in SEEDS:
    out = f'{RAW}/stl10_p1_lev_seed{s}.json'
    if os.path.exists(out): print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset stl10 --patch_size 1 --epochs 50 --autobatch --grad_clip 1.0 --kappa_star 0'
           f' --seed {s} --backends flash_nystrom_vanilla flash_nystrom_leverage flash_nystrom_leverage_det'
           f' --out_json {out} 2>&1 | tee {RAW}/stl10_p1_lev_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}
print('stl@9216 done')

## 9. Aggregate -> mean +/- std + A/B deltas  ->  `aggregated.json`
Rolls every `raw/*.json` into per-(experiment, backend) mean +/- std, and prints the
segment -> leverage delta per experiment.

In [ ]:
import glob, json, statistics as st
def agg(v):
    v = [x for x in v if x is not None]
    return (st.mean(v), st.pstdev(v) if len(v) > 1 else 0.0, len(v)) if v else (float('nan'), 0.0, 0)

# vision: train_three_way writes {..., results:[{label,test_acc}]}
vis = {}
for f in glob.glob(f'{RAW}/cifar*_seed*.json') + glob.glob(f'{RAW}/stl10_*_seed*.json'):
    d = json.load(open(f)); exp = os.path.basename(f).split('_seed')[0]
    for r in d['results']:
        vis.setdefault((exp, r['label']), []).append(r['test_acc'])
print('VISION (test acc %, mean +/- std):')
for (e, b), a in sorted(vis.items()):
    mu, sd, n = agg(a); print(f'  {e:16s} {b:18s}: {mu:5.1f} +/- {sd:.1f} (n={n})')

# mqar: flat {backend, recall}
mq = {}
for f in glob.glob(f'{RAW}/mqar_*_seed*.json'):
    d = json.load(open(f)); mq.setdefault(d['backend'], []).append(d.get('recall'))
print('\nMQAR (recall %, mean +/- std):')
for b, v in sorted(mq.items()):
    mu, sd, n = agg(v); print(f'  {b:24s}: {mu:5.2f} +/- {sd:.2f} (n={n})')

# A/B deltas: FlashNystrom-V (segment) -> FlashNystrom-Lev (leverage)
print('\nA/B DELTA (leverage - segment):')
exps = sorted({e for (e, _) in vis})
for e in exps:
    seg = agg(vis.get((e, 'FlashNystrom-V'), []))[0]
    lev = agg(vis.get((e, 'FlashNystrom-Lev'), []))[0]
    if seg == seg and lev == lev:  # not NaN
        print(f'  {e:16s}: segment {seg:5.1f} -> leverage {lev:5.1f}  (delta {lev-seg:+.1f})')
seg_m = agg(mq.get('flash_nystrom', []))[0]; lev_m = agg(mq.get('flash_nystrom_leverage', []))[0]
if seg_m == seg_m and lev_m == lev_m:
    print(f'  {"mqar":16s}: segment {seg_m:5.1f} -> leverage {lev_m:5.1f}  (delta {lev_m-seg_m:+.1f})')

json.dump({'vision': {f'{e}|{b}': agg(a) for (e, b), a in vis.items()},
           'mqar':   {b: agg(v) for b, v in mq.items()}},
          open(f'{OUTDIR}/aggregated.json', 'w'), indent=2)
print('\nwrote', OUTDIR + '/aggregated.json')

## Notes
- Every training cell is resumable: delete a `raw/*.json` to force that run to re-execute.
- All arms use `kappa_star=0` (paper default; the ridge does not help these tasks) so the A/B
  isolates the landmark selector alone.
- `FlashNystrom-V` = segment-mean, `FlashNystrom-Lev` = leverage-Voronoi (`landmark_mode=1`).
- **Batch size is identical across the arms of each experiment** -- required so the accuracy gap is
  attributable to the landmark selector, not to batch size (which changes gradient noise / effective LR).
  Vision uses coordinated `--autobatch`: it probes the max batch EACH arm can hold, then trains ALL arms
  at the MIN of those maxima (largest batch that fits every arm; the memory-hungry reference arm is
  usually binding). MQAR uses the validated fixed `--batch_size 256` recipe (its LR/epochs are
  calibrated to batch 256; autobatch would also rescale epochs, so fixing it keeps the recipe intact).
- Local runs: set `SEEDS=[0]`, and drop the STL@9216 cell / lower `--autobatch_cap` if VRAM is tight.